# Step 6 – Prognostic validation (Section 5.6, Tables 10–12)

Genomic-grade score, the four published genes and grade 2 re-stratification by the three-class model.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

In [ ]:
import pickle, numpy as np, pandas as pd, warnings
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from sksurv.metrics import cumulative_dynamic_auc
%run pipeline.ipynb

warnings.filterwarnings("ignore")
S = pickle.load(open("summary.pkl", "rb"))
A = S["A"]
CG = list(pd.read_csv("common_genes.csv").iloc[:, 0])
C4 = ["GSE7390", "UPP", "MAINZ", "UNT"]
END = {"GSE7390": "rfs", "UPP": "rfs", "MAINZ": "dmfs", "UNT": "rfs"}
PUB = ["E2F3", "PSMC3IP", "GINS1", "PLAGL2"]
frames = []
for c in C4:
    X, y, P = load(c, genes=CG, zscore=True)
    e = END[c]
    df = pd.DataFrame({"cohort": c, "grade": y + 1, "time": P[f"t.{e}"].values / 365.25, "event": P[f"e.{e}"].values,
                       "age": P["age"].values / 10, "size": P["size"].values, "er": P["er"].values, "node": P["node"].values},
                      index=X.index)
    pr = A[c]["proba_oof"]
    df["pG1"], df["pG2"], df["pG3"] = pr[:, 0], pr[:, 1], pr[:, 2]
    df["ggs"] = pr[:, 2] - pr[:, 0]           # HWNB genomic-grade score (out-of-fold)
    for g in PUB:
        df[g] = X[g].values
    df["pub4"] = X[PUB].mean(1).values        # unweighted published four-gene score
    frames.append(df)
D = pd.concat(frames)
D = D.dropna(subset=["time", "event"])
D["event"] = D["event"].astype(int)
D["g3"] = (D["grade"] == 3).astype(int)
D["size_gt2"] = (D["size"] > 2).astype(int)
out = {}

In [ ]:
def cox(df, cols, strata=None):
    d = df[["time", "event"] + cols + ([strata] if strata else [])].dropna()
    m = CoxPHFitter().fit(d, "time", "event", strata=[strata] if strata else None)
    s = m.summary
    return {v: (s.loc[v, "exp(coef)"], s.loc[v, "exp(coef) lower 95%"], s.loc[v, "exp(coef) upper 95%"], s.loc[v, "p"]) for v in cols}, len(d), int(d["event"].sum()), m

In [ ]:
def fmt(t):
    return f"{t[0]:.2f} ({t[1]:.2f}–{t[2]:.2f}), p={t[3]:.3g}"

### 1. GSE7390: HWNB genomic-grade score, out-of-fold

In [ ]:
# ---- 1. GSE7390: HWNB genomic-grade score, out-of-fold
T = D[D.cohort == "GSE7390"].copy()
T["ggs_high"] = (T["ggs"] > T["ggs"].median()).astype(int)
ci = concordance_index(T["time"], -T["ggs"], T["event"])
st = np.array([(bool(e), t) for e, t in zip(T["event"], T["time"])], dtype=[("e", bool), ("t", float)])
auc, _ = cumulative_dynamic_auc(st, st, T["ggs"].values, [5, 10])
uni, n, ev, _ = cox(T, ["ggs_high"])
multi, n2, ev2, m = cox(T, ["ggs_high", "g3", "er", "size_gt2", "age"])
ph = m.check_assumptions(T[["time", "event", "ggs_high", "g3", "er", "size_gt2", "age"]].dropna(), show_plots=False, p_value_threshold=0.01) if False else None
out["gse7390_ggs"] = {"cindex": ci, "auc5": auc[0], "auc10": auc[1], "uni": uni, "multi": multi, "n": n2, "events": ev2}
print("GSE7390 GGS C-index %.3f  tdAUC5 %.3f tdAUC10 %.3f" % (ci, auc[0], auc[1]), "n", n2, "events", ev2)
print("  univariable", fmt(uni["ggs_high"]))
for k, v in multi.items(): print("  multi", k, fmt(v))
from lifelines.statistics import proportional_hazard_test
pht = proportional_hazard_test(m, T[["time", "event", "ggs_high", "g3", "er", "size_gt2", "age"]].dropna(), time_transform="rank")
print("  PH test min p", pht.summary["p"].min().round(3))
out["ph_min_p"] = pht.summary["p"].min()

### 2. external cohorts: C-index of out-of-fold GGS and pooled stratified Cox

In [ ]:
# ---- 2. external cohorts: C-index of out-of-fold GGS and pooled stratified Cox
for c in C4:
    d = D[D.cohort == c]
    out[f"ci_{c}"] = concordance_index(d["time"], -d["ggs"], d["event"])
    out[f"ci_pub4_{c}"] = concordance_index(d["time"], -d["pub4"], d["event"])
    out[f"n_{c}"] = (len(d), int(d["event"].sum()))
print("C-index GGS", {c: round(out[f"ci_{c}"], 3) for c in C4}, "\nC-index pub4", {c: round(out[f"ci_pub4_{c}"], 3) for c in C4}, {c: out[f"n_{c}"] for c in C4})
D["ggs_high"] = D.groupby("cohort")["ggs"].transform(lambda s: (s > s.median()).astype(int))
pool_multi, pn, pe, _ = cox(D, ["ggs_high", "g3", "er", "size_gt2", "age", "node"], strata="cohort")
out["pool_multi"] = (pool_multi, pn, pe)
print("Pooled stratified multivariable (n=%d, events=%d)" % (pn, pe)); [print("  ", k, fmt(v)) for k, v in pool_multi.items()]

### 3. published four genes: per-SD hazard ratios, pooled stratified, univariable and adjusted

In [ ]:
# ---- 3. published four genes: per-SD hazard ratios, pooled stratified, univariable and adjusted
pubres = {}
for g in PUB + ["pub4"]:
    u, _, _, _ = cox(D, [g], strata="cohort")
    a, _, _, _ = cox(D, [g, "g3", "er", "size_gt2", "age", "node"], strata="cohort")
    pubres[g] = (u[g], a[g])
    print("PUB", g, "uni", fmt(u[g]), "| adj", fmt(a[g]))
out["pub"] = pubres
# within-grade (grade 2) effect of GINS1 and E2F3, pooled
for g in ["GINS1", "E2F3"]:
    u, n_, e_, _ = cox(D[D.grade == 2], [g], strata="cohort")
    out[f"g2_{g}"] = (u[g], n_, e_)
    print("grade2", g, fmt(u[g]), n_, e_)

### 4. grade 2 re-stratification by HWNB probabilities (out-of-fold)

In [ ]:
# ---- 4. grade 2 re-stratification by HWNB probabilities (out-of-fold)
G2 = D[D.grade == 2].copy()
G2["g3like"] = (G2["pG3"] > G2["pG1"]).astype(int)
res = {}
for name, d in [("GSE7390", G2[G2.cohort == "GSE7390"]), ("pooled", G2)]:
    lr = logrank_test(d.time[d.g3like == 1], d.time[d.g3like == 0], d.event[d.g3like == 1], d.event[d.g3like == 0])
    km = {}
    for grp in [0, 1]:
        k = KaplanMeierFitter().fit(d.time[d.g3like == grp], d.event[d.g3like == grp])
        km[grp] = (int((d.g3like == grp).sum()), float(k.survival_function_at_times(5).iloc[0]), float(k.survival_function_at_times(10).iloc[0]))
    cols = ["g3like", "er", "size_gt2", "age"] + (["node"] if name == "pooled" else [])
    adj, n_, e_, _ = cox(d, cols, strata="cohort" if name == "pooled" else None)
    res[name] = {"logrank_p": lr.p_value, "km": km, "adj": adj["g3like"], "n": n_, "events": e_}
    print("G2 restrat", name, "logrank p=%.4g" % lr.p_value, km, "adjHR", fmt(adj["g3like"]), n_, e_)
# KM curves for the pooled figure
curves = {}
for grp in [0, 1]:
    d = G2[G2.g3like == grp]
    k = KaplanMeierFitter().fit(d.time, d.event)
    sf = k.survival_function_
    curves[grp] = [(float(t), float(v)) for t, v in zip(sf.index, sf.iloc[:, 0]) if t <= 12]
res["curves"] = curves
# compare to G1 and G3 tumours (pooled) for context
for gr in [1, 3]:
    d = D[D.grade == gr]; k = KaplanMeierFitter().fit(d.time, d.event)
    res[f"grade{gr}_5y"] = float(k.survival_function_at_times(5).iloc[0])
    res[f"curve_g{gr}"] = [(float(t), float(v)) for t, v in zip(k.survival_function_.index, k.survival_function_.iloc[:, 0]) if t <= 12]
print("pooled 5y grade1 %.3f grade3 %.3f" % (res["grade1_5y"], res["grade3_5y"]))
out["restrat"] = res
pickle.dump(out, open("survival.pkl", "wb"))